In [1]:
import pandas as pd
import os

# setwd to parent directory
os.chdir(os.path.dirname(os.getcwd()))

ANNOT_FILE = "data/dataset_092624.xlsx"

In [2]:
# Load the xlsx file into a dataframe
annotations = pd.read_excel(ANNOT_FILE)

# Display the head of the dataframe
print(annotations.head())

# Filter only relevant rows : valid_yn = 'yes' and source = 'dryad'
annotations = annotations[(annotations['valid_yn'] == 'yes') & (annotations['source'] == 'dryad')]
annotations.head()

   id                                      url  \
0   3  https://doi.org/10.5061/dryad.05qfttf0n   
1   4  https://doi.org/10.5061/dryad.0rxwdbs2c   
2   5      https://doi.org/10.5061/dryad.121sk   
3   7  https://doi.org/10.5061/dryad.12jm63xtw   
4   9      https://doi.org/10.5061/dryad.1771t   

                                               title  \
0  Fish mock community with 41 species from 13 or...   
1  Exploration and diet specialization in eastern...   
2  Data from: Paternity analysis of wood turtles ...   
3  Data from: 60 specific eDNA qPCR assays to det...   
4  Data from: Resampling method for applying dens...   

                                           full_text publication_year  source  \
0  Fish mock community with 41 species from 13 or...             2020  zenodo   
1  Exploration and diet specialization in eastern...             2022  zenodo   
2  Data from: Paternity analysis of wood turtles ...             2017  zenodo   
3  Data from: 60 specific eDNA qPCR as

,id,url,title,full_text,publication_year,source,id_query,reason_non_valid,valid_yn,dataset_relevance,...,threatened_species,new_species_science,new_species_region,bias_north_south,url.1,MC_dataset_type,MC_spatial_range,MC_temporal_range,MC_relevance,MC_relevance_modifiers
4,9,https://doi.org/10.5061/dryad.1771t,Data from: Resampling method for applying dens...,Data from: Resampling method for applying dens...,2016,dryad,"0,3,4,8,10",NaN,yes,L,...,False,False,False,False,https://doi.org/10.5061/dryad.1771t,H,X,L,L,L
8,13,https://doi.org/10.5061/dryad.24rj8,"Data from: Aspicilia bicensis (Megasporaceae),...","Data from: Aspicilia bicensis (Megasporaceae),...",2016,dryad,"3,5,8",NaN,yes,M,...,False,True,False,False,https://doi.org/10.5061/dryad.24rj8,L,X,X,X,L
9,16,https://doi.org/10.5061/dryad.2b8f1,Data from: Do genetic drift and accumulation o...,Data from: Do genetic drift and accumulation o...,2017,dryad,"3,4,6",NaN,yes,H,...,False,False,False,True,https://doi.org/10.5061/dryad.2b8f1,H,H,M,H,H
11,19,https://doi.org/10.5061/dryad.2n5h6,Data from: The genetic signature of range expa...,Data from: The genetic signature of range expa...,2016,dryad,"9,7,8,3,6",NaN,yes,M,...,False,False,False,False,https://doi.org/10.5061/dryad.2n5h6,H,L,M,M,M
15,27,https://doi.org/10.5061/dryad.3nh72,Data from: Patchy distribution and low effecti...,Data from: Patchy distribution and low effecti...,2017,dryad,"3,6",NaN,yes,M,...,True,False,False,False,https://doi.org/10.5061/dryad.3nh72,H,X,M,L,M


In [13]:
from llm_metadata.prefect_pipeline import doi_classification_pipeline

# Run classification on the first 3 rows
dois = annotations['url'].head(3)
dois = [doi.replace('https://doi.org/', '') for doi in dois]

classification_results = doi_classification_pipeline(dois=dois)


10:11:30.462 | INFO    | Flow run 'myrtle-tiger' - Beginning flow run 'myrtle-tiger' for flow 'doi-classification-pipeline'

10:11:31.514 | INFO    | Task run 'fetch_abstracts-415' - Finished in state Completed()

10:11:33.456 | INFO    | Task run 'classify_abstract_task-38f' - Finished in state Completed()

10:11:33.763 | INFO    | Task run 'classify_abstract_task-41a' - Finished in state Completed()

10:11:34.252 | INFO    | Task run 'classify_abstract_task-ee1' - Finished in state Completed()

10:11:34.323 | INFO    | Flow run 'myrtle-tiger' - Finished in state Completed()

In [33]:
classification_results[0]['output'].model_dump()

{'data_type': [<EBVDataType.TRAITS: 'traits'>,
  <EBVDataType.GENETIC_ANALYSIS: 'genetic_analysis'>],
 'geospatial_info_dataset': <GeospatialInfoType.SITE: 'site'>,
 'spatial_range_km2': None,
 'temporal_range': None,
 'temp_range_i': None,
 'temp_range_f': None,
 'taxons': 'Aspicilia bicensis',
 'referred_dataset': 'Parc national du Bic, Quebec, Canada'}

In [ ]:
outputs = [r['output'].dict() for r in classification_results]
for d, doi in zip(outputs, dois):
    d.update({'doi': doi})

# Parse outputs to pandas
output_df = pd.DataFrame(outputs)

# Convert enums to their string representations
output_df['data_type'] = output_df['data_type'].apply(lambda xl: [x.value if hasattr(x, 'value') else x for x in xl])
output_df['geospatial_info_dataset'] = output_df['geospatial_info_dataset'].apply(lambda x: x.value if hasattr(x, 'value') else x)

# Input df with only attributes that are present in the output
compared_df = annotations[output_df.columns].head()


/tmp/ipykernel_113130/283596892.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  outputs = [r['output'].dict() for r in classification_results]


,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,taxons,referred_dataset,doi
0,"[traits, genetic_analysis]",site,None,None,None,None,Aspicilia bicensis,"Parc national du Bic, Quebec, Canada",10.5061/dryad.1771t
1,"[abundance, distribution]",site,None,None,None,None,"raccoons, striped skunks",None,10.5061/dryad.24rj8
2,"[genetic_analysis, traits]",site,None,None,None,None,Lake Trout (Salvelinus namaycush),"Québec, Canada",10.5061/dryad.2b8f1


In [60]:
compared_df = annotations[['title', 'url', 'data_type', 'geospatial_info_dataset', 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 'species', 'referred_dataset']].head(3)
compared_df.rename(columns={'species': 'taxons'}, inplace=True)
compared_df['doi'] = compared_df['url'].apply(lambda x: x.replace('https://doi.org/', ''))
compared_df['data_type'] = compared_df['data_type'].apply(lambda x: x.split(','))
compared_df['classification'] = 'manual'

output_df['classification'] = 'automated'

# Append the output_df to the compared_df
compared_df = pd.concat([compared_df, output_df], ignore_index=True)

# Reorder the columns to put doi first
reorder_columns = [c for c in compared_df.columns.tolist() if c not in ('title', 'url', 'doi', 'classification')]
compared_df = compared_df[['title', 'url', 'doi', 'classification'] + reorder_columns]

# Order by doi, classification
compared_df = compared_df.sort_values(by=['doi', 'classification'], ascending=[True, False])
compared_df


,title,url,doi,classification,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,taxons,referred_dataset
0,Data from: Resampling method for applying dens...,https://doi.org/10.5061/dryad.1771t,10.5061/dryad.1771t,manual,[density],sample coordinates,not given,2008,2008,2008,"raccoons, striped skunks",NaN
3,NaN,NaN,10.5061/dryad.1771t,automated,"[traits, genetic_analysis]",site,None,None,None,None,Aspicilia bicensis,"Parc national du Bic, Quebec, Canada"
1,"Data from: Aspicilia bicensis (Megasporaceae),...",https://doi.org/10.5061/dryad.24rj8,10.5061/dryad.24rj8,manual,[genetic analysis],not given,not given,NaN,NaN,NaN,Aspicilia bicensis,NaN
4,NaN,NaN,10.5061/dryad.24rj8,automated,"[abundance, distribution]",site,None,None,None,None,"raccoons, striped skunks",None
2,Data from: Do genetic drift and accumulation o...,https://doi.org/10.5061/dryad.2b8f1,10.5061/dryad.2b8f1,manual,[EBV genetic analysis],not given,500000,2003-2013,2003,2013,Salvelinus namaycush,NaN
5,NaN,NaN,10.5061/dryad.2b8f1,automated,"[genetic_analysis, traits]",site,None,None,None,None,Lake Trout (Salvelinus namaycush),"Québec, Canada"


In [71]:
def compare_label(truth, pred):
    if truth and truth == pred:
        return 'TP'
    elif truth and truth != pred:
        return 'FN'
    elif not truth and pred:
        return 'FP'
    else:
        return 'TN'

# Fuzzy matching
from fuzzywuzzy import fuzz
def fuzzy_compare_label(truth, pred):
    fuzz_ratio = fuzz.ratio(truth, pred)
    if truth and fuzz_ratio > 80:
        return 'TP'
    elif truth and fuzz_ratio <= 80:
        return 'FN'
    elif not truth and pred:
        return 'FP'
    else:
        return 'TN'
    
def compare_multilabel(truth: list, pred: list):
    out = []
    for p in pred:
        if truth and p in truth:
            out.append('TP')
        elif truth and p not in truth:
            out.append('FN')
        elif not truth and p:
            out.append('FP')
        else:
            out.append('TN')
    return out

FEATURE_COMPARE_FN = {
    'data_type': compare_multilabel,
    'geospatial_info_dataset': compare_label,
    'spatial_range_km2': compare_label,
    'temporal_range': compare_label,
    'temp_range_i': compare_label,
    'temp_range_f': compare_label,
    'taxons': fuzzy_compare_label,
    'referred_dataset': compare_label
}

def compare_doi_rows(df, doi) -> pd.Series:
    pred_series = df[(df['doi'] == doi) & (df['classification'] == 'automated')][0]
    truth_series = df[(df['doi'] == doi) & (df['classification'] == 'manual')][0]
    if pred_series.empty:
        return None
    
    out = {}
    for feature, compare_fn in FEATURE_COMPARE_FN.items():
        out[feature] = compare_fn(truth_series[feature], pred_series[feature])
    return pd.Series(out)

dois = compared_df['doi'].unique().tolist()

compare_doi_rows(compared_df, '10.5061/dryad.1771t')

KeyError: 0

In [69]:
compared_df['doi'].unique().tolist()

['10.5061/dryad.1771t', '10.5061/dryad.24rj8', '10.5061/dryad.2b8f1']